# Week 6 — ROI localizer for 3D fire positioning

The upstream detector remains responsible for fire/no-fire classification. This notebook trains only the narrow-scene ROI point refiner and writes its checkpoint to `/kaggle/working/week6_roi`.

In [ ]:
from pathlib import Path
import subprocess, sys

CODE_CANDIDATES = [Path('/kaggle/input/sam-experiment-code'), Path('/kaggle/input/sam-experiment-code/sam-experiment-code')]
CODE_DIR = next((p for p in CODE_CANDIDATES if (p / 'train_roi_localizer.py').exists()), None)
if CODE_DIR is None:
    raise FileNotFoundError('Upload sam-experiment-code.zip or the sam-experiment-code folder as a Kaggle dataset.')
sys.path.insert(0, str(CODE_DIR))
WORK_DIR = Path('/kaggle/working/week6_roi')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('CODE_DIR =', CODE_DIR)

In [ ]:
def first_existing(paths):
    return next((p for p in paths if p.exists()), None)

LABELS = first_existing([
    Path('/kaggle/input/fire-model-data/dataset_labels (1).json'),
    Path('/kaggle/input/sam-experiment-code/fire-model-data/dataset_labels (1).json'),
])
MODEL = first_existing([
    Path('/kaggle/input/fire-model-data/best.pth'),
    Path('/kaggle/input/sam-experiment-code/fire-model-data/best.pth'),
])
DATASET_ROOT = first_existing([
    Path('/kaggle/input/fire-detection-from-cctv'),
    Path('/kaggle/input/sam-experiment-code/fire-detection-from-cctv'),
])
for name, value in {'LABELS': LABELS, 'MODEL': MODEL, 'DATASET_ROOT': DATASET_ROOT}.items():
    if value is None:
        raise FileNotFoundError(f'Missing {name}; attach the matching Kaggle input dataset.')
print(LABELS, MODEL, DATASET_ROOT)

## Optional: create coarse points from the existing detector

This uses `best.pth` only as the upstream detector and never overwrites it. If another detector is used in deployment, replace this manifest with its predictions.

In [ ]:
COARSE = WORK_DIR / 'coarse_manifest.json'
cmd = [sys.executable, str(CODE_DIR / 'build_coarse_manifest.py'), '--labels', str(LABELS), '--dataset-root', str(DATASET_ROOT), '--model', str(MODEL), '--output', str(COARSE)]
subprocess.run(cmd, check=True)
print(COARSE)

In [ ]:
cmd = [sys.executable, str(CODE_DIR / 'train_roi_localizer.py'), '--labels', str(LABELS), '--dataset-root', str(DATASET_ROOT), '--init-checkpoint', str(MODEL), '--coarse-manifest', str(COARSE), '--output-dir', str(WORK_DIR), '--epochs', '30', '--batch-size', '32', '--workers', '2', '--freeze-epochs', '5']
subprocess.run(cmd, check=True)

## Result

Use `best_roi.pth` in `main_localization.py`. It contains an ROI-only spatial localizer, not a replacement detector.